# 02.02 — Evidence adaptation

Câu hỏi: adaptation của checkpoint đã khai báo trước thay đổi kết quả ra sao? Notebook không train; nó đọc experiment record và evidence của cùng checkpoint trên IT Validation và General Validation. Mỗi checkpoint chỉ được xem là đầy đủ khi có cả hai run.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

def find_root(start):
    for candidate in (start, *start.parents):
        if (candidate / 'configs/phase02_protocol.json').is_file():
            return candidate
    raise FileNotFoundError('Không tìm thấy repository root chứa configs/phase02_protocol.json.')

ROOT = find_root(Path.cwd())
BASE = ROOT / 'evidence/phase02'
EXPECTED_DATASETS = {'selection_validation', 'general_validation'}
rows, incomplete, training_rows = [], [], []
seen_training_runs = set()
for metric_path in BASE.glob('*/*/adapted/*/*/*/metrics.json'):
    run_path = metric_path.with_name('run.json')
    record_path = metric_path.parents[2] / 'experiment_record.md'
    training_path = record_path.with_name('training_run.json')
    if not run_path.is_file() or not record_path.is_file() or not training_path.is_file():
        incomplete.append({'evidence': str(metric_path.relative_to(ROOT)), 'missing_run_json': not run_path.is_file(), 'missing_experiment_record': not record_path.is_file(), 'missing_training_run': not training_path.is_file()})
        continue
    metric = json.loads(metric_path.read_text(encoding='utf-8'))
    run = json.loads(run_path.read_text(encoding='utf-8'))
    training = json.loads(training_path.read_text(encoding='utf-8'))
    if str(training_path) not in seen_training_runs:
        config = training['training_configuration']
        training_rows.append({'model': training['model']['key'], 'direction': training['direction'], 'experiment_id': training['experiment_id'], 'checkpoint_id': training['checkpoint_id'], 'adaptation_method': training['adaptation_method']['name'], 'lora': training['adaptation_method'].get('lora'), 'train_rows': training['train_data']['rows'], 'epochs': config['epochs'], 'effective_batch_size': config['effective_batch_size'], 'learning_rate': config['learning_rate'], 'seed': config['seed'], 'training_run': str(training_path.relative_to(ROOT))})
        seen_training_runs.add(str(training_path))
    rows.append({'model': run['adapter']['model_key'], 'direction': run['adapter']['direction'], 'experiment_id': metric_path.parents[2].name, 'checkpoint_id': metric_path.parents[1].name, 'dataset': metric['dataset_role'], 'rows': metric['rows'], 'chrF++': metric['chrF++'], 'SacreBLEU': metric['sacreBLEU'], 'experiment_record': str(record_path.relative_to(ROOT)), 'training_run': str(training_path.relative_to(ROOT)), 'evidence': str(metric_path.parent.relative_to(ROOT))})

adapted = pd.DataFrame(rows)
if adapted.empty:
    display(pd.DataFrame({'status': ['Chưa có adapted evaluation hoàn tất.']}))
else:
    display(pd.DataFrame(training_rows).sort_values(['direction', 'experiment_id', 'checkpoint_id']))
    adapted = adapted.sort_values(['direction', 'experiment_id', 'checkpoint_id', 'dataset'])
    display(adapted)
    coverage = (adapted.groupby(['model', 'direction', 'experiment_id', 'checkpoint_id'], as_index=False)['dataset']
                .agg(lambda values: ', '.join(sorted(set(values))))
                .rename(columns={'dataset': 'validation_sets'}))
    coverage['complete_two_validation_sets'] = coverage['validation_sets'].map(lambda value: set(value.split(', ')) == EXPECTED_DATASETS)
    display(coverage)

    pretrained_rows = []
    for metric_path in BASE.glob('*/*/pretrained/*/metrics.json'):
        run_path = metric_path.with_name('run.json')
        if not run_path.is_file():
            continue
        metric = json.loads(metric_path.read_text(encoding='utf-8'))
        run = json.loads(run_path.read_text(encoding='utf-8'))
        pretrained_rows.append({'model': run['adapter']['model_key'], 'direction': run['adapter']['direction'], 'dataset': metric['dataset_role'], 'pretrained_chrF++': metric['chrF++'], 'pretrained_SacreBLEU': metric['sacreBLEU']})
    comparison = adapted.merge(pd.DataFrame(pretrained_rows), on=['model', 'direction', 'dataset'], how='left')
    display(comparison[['model', 'direction', 'experiment_id', 'checkpoint_id', 'dataset', 'pretrained_chrF++', 'chrF++', 'pretrained_SacreBLEU', 'SacreBLEU']])

if incomplete:
    display(pd.DataFrame(incomplete))

,model,direction,experiment_id,checkpoint_id,adaptation_method,lora,train_rows,epochs,effective_batch_size,learning_rate,seed,training_run
0,envit5,en_to_vi,envit5-en-to-vi-lora-v1,final-epoch-1,lora,"{'rank': 8, 'alpha': 16, 'dropout': 0.0, 'targ...",121547,1.0,8,0.0002,42,evidence\phase02\envit5\en_to_vi\adapted\envit...
1,envit5,vi_to_en,envit5-vi-to-en-lora-v1,final-epoch-1,lora,"{'rank': 8, 'alpha': 16, 'dropout': 0.0, 'targ...",121547,1.0,8,0.0002,42,evidence\phase02\envit5\vi_to_en\adapted\envit...


,model,direction,experiment_id,checkpoint_id,dataset,rows,chrF++,SacreBLEU,experiment_record,training_run,evidence
0,envit5,en_to_vi,envit5-en-to-vi-lora-v1,final-epoch-1,general_validation,997,60.364459,39.238366,evidence\phase02\envit5\en_to_vi\adapted\envit...,evidence\phase02\envit5\en_to_vi\adapted\envit...,evidence\phase02\envit5\en_to_vi\adapted\envit...
1,envit5,en_to_vi,envit5-en-to-vi-lora-v1,final-epoch-1,selection_validation,15253,71.174877,57.762843,evidence\phase02\envit5\en_to_vi\adapted\envit...,evidence\phase02\envit5\en_to_vi\adapted\envit...,evidence\phase02\envit5\en_to_vi\adapted\envit...
2,envit5,vi_to_en,envit5-vi-to-en-lora-v1,final-epoch-1,general_validation,997,58.449516,29.842579,evidence\phase02\envit5\vi_to_en\adapted\envit...,evidence\phase02\envit5\vi_to_en\adapted\envit...,evidence\phase02\envit5\vi_to_en\adapted\envit...
3,envit5,vi_to_en,envit5-vi-to-en-lora-v1,final-epoch-1,selection_validation,15253,65.383472,41.319147,evidence\phase02\envit5\vi_to_en\adapted\envit...,evidence\phase02\envit5\vi_to_en\adapted\envit...,evidence\phase02\envit5\vi_to_en\adapted\envit...


,model,direction,experiment_id,checkpoint_id,validation_sets,complete_two_validation_sets
0,envit5,en_to_vi,envit5-en-to-vi-lora-v1,final-epoch-1,"general_validation, selection_validation",True
1,envit5,vi_to_en,envit5-vi-to-en-lora-v1,final-epoch-1,"general_validation, selection_validation",True


,model,direction,experiment_id,checkpoint_id,dataset,pretrained_chrF++,chrF++,pretrained_SacreBLEU,SacreBLEU
0,envit5,en_to_vi,envit5-en-to-vi-lora-v1,final-epoch-1,general_validation,59.822968,60.364459,38.398872,39.238366
1,envit5,en_to_vi,envit5-en-to-vi-lora-v1,final-epoch-1,selection_validation,64.981633,71.174877,48.974620,57.762843
2,envit5,vi_to_en,envit5-vi-to-en-lora-v1,final-epoch-1,general_validation,60.143713,58.449516,32.063374,29.842579
3,envit5,vi_to_en,envit5-vi-to-en-lora-v1,final-epoch-1,selection_validation,60.431968,65.383472,31.983891,41.319147


Bảng thứ hai xác nhận mỗi checkpoint có đủ hai validation set. Bảng cuối đối chiếu pretrained và adapted theo cùng model, chiều dịch và dataset. Chênh lệch metric chỉ là evidence để người thực hiện đánh giá; notebook không tự chọn checkpoint hoặc Core.